# **LAPORAN TUGAS PROJECT PENGENALAN POLA**

**Judul Project:** Analisis Prediktif Fluktuasi Harga Komoditas Pangan di Indonesia Menggunakan PCA LDA dan Random Forest  
**Mata Kuliah:** Pengenalan Pola  
**Dosen Pengampu:** Prof. I Gede Pasek Suta Wijaya  
**Disusun Oleh:**
1. Nengah Anggi Juwita Sari - F1D02310021
2. Naufal Ihsanul Islam - F1D02310084
3. Lutfi Alfarizi - F1D02310121  

**Program Studi:** Teknik Informatika FT-UNRAM  
**Tahun:** 2026

**Summary**

Penelitian ini membandingkan empat skenario pemodelan untuk meramal fluktuasi harga komoditas pangan di 38 provinsi Indonesia, yaitu ARIMA (baseline univariat), Random Forest Murni, PCA + Random Forest, dan LDA + Random Forest. Dataset bersumber dari SP2KP Kementerian Perdagangan RI dengan total 23.372 baris data periode 2021–2025. Setiap skenario dievaluasi melalui tiga fase berjenjang untuk membongkar kecacatan metodologi umum: Fase 1 menyingkap ilusi akurasi akibat *data leakage* (split acak) atau agregasi rata-rata nasional yang menyamarkan fluktuasi lokal; Fase 2 menerapkan evaluasi kronologis murni (data 2021–2024 untuk latih, 2025 untuk uji) pada level provinsi-komoditas; dan Fase 3 menghilangkan bias skala harga dengan menghitung metrik secara independen per komoditas sebelum dirata-ratakan. Hasil pada Fase 3 menunjukkan bahwa Random Forest Murni memberikan performa terbaik dengan RMSE Rp 2.548 dan $R^2$ 79,28%, mengungguli ARIMA ($R^2$ 58,86%), PCA + Random Forest ($R^2$ 35,46%), dan LDA + Random Forest ($R^2$ 14,70%). Temuan ini membuktikan bahwa rekayasa fitur tambahan (lag, rolling statistics, fitur musiman) jauh lebih bermanfaat bagi performa prediksi dibandingkan reduksi dimensi melalui PCA maupun LDA, yang justru membuang informasi kontinuitas temporal dan spasial yang krusial bagi peramalan deret waktu harga pangan.

**Keyword:** Peramalan Harga Pangan, Random Forest, ARIMA, PCA, LDA, Data Leakage, Time-Series, Reduksi Dimensi, Bias Skala, Pengenalan Pola

## 1. PENDAHULUAN

### 1.1 Latar Belakang
Fluktuasi harga komoditas pangan di 38 provinsi adalah isu kritis yang perlu dipetakan secara matematis untuk perumusan kebijakan publik. Seringkali, model prediktif dievaluasi secara keliru (mengalami *Data Leakage*) yang menghasilkan ilusi akurasi 100% palsu. Oleh karena itu, proyek ini membandingkan kemampuan nyata dari arsitektur Random Forest, PCA, LDA, dan ARIMA dalam memprediksi fluktuasi deret waktu secara murni dan kronologis.

### 1.2 Rumusan Masalah
1. Bagaimana dampak penerapan reduksi dimensi (PCA dan LDA) terhadap data *time-series* komoditas pangan?
2. Bagaimana perbandingan performa sejati *Random Forest Regressor* dan ARIMA dalam meramal harga setelah metrik dievaluasi murni tanpa Bias Skala?

### 1.3 Tujuan Proyek
1. Menerapkan metode ARIMA, RF Murni, PCA+RF, dan LDA+RF untuk memetakan pola pergerakan harga pangan daerah.
2. Menganalisis tingkat akurasi nyata (RMSE, MAE, MAPE, dan $R^2$) untuk membuktikan dan menggugurkan metrik manipulatif akibat ilusi bocor data.

## 2. DATASET DAN PRA-PROSES DATA (PREPROCESSING)

### 2.1 Deskripsi Dataset
* **Sumber Data:** Dataset yang digunakan dalam penelitian ini bersumber dari Sistem Pemantauan Pasar dan Kebutuhan Pokok (SP2KP) Kementerian Perdagangan Republik Indonesia. Dataset ini tersedia secara publik dan digunakan secara resmi oleh pemerintah sebagai alat pemantauan harga pangan nasional.
* **Jumlah Data:** Total sampel adalah 23.372 baris untuk semua model.
  * **Model selain ARIMA:** Menggunakan data latih (*train*) sebanyak 14.661 baris dan data uji (*test*) sebanyak 8.711 baris.
  * **Model ARIMA:** Data latih menggunakan semua data dari tahun 2021 hingga 2024, dan data uji menggunakan data tahun 2025.
* **Fitur/Atribut:** Kolom asli pada dataset terdiri dari `Nama Provinsi`, `Komoditas`, `Tahun`, `Bulan`, dan `Harga`.
  * **Pada model ARIMA:** Hanya menggunakan kolom `Harga_Numerik`, yaitu kolom yang isinya sudah dibersihkan dan angkanya dijadikan format numerik (bukan string).
  * **Pada model selain ARIMA:** Menggunakan tambahan kolom hasil *feature engineering*, yaitu `Bulan_Sin`, `Bulan_Cos`, `Ramadan_Lebaran`, `Rolling_Mean_3`, `Rolling_Std_3`, dan `Lag_1` hingga `Lag_12`.

## 3. METODOLOGI DAN ARSITEKTUR MODEL

### 3.1 Ekstraksi Fitur (Feature Extraction)
* **Linear Discriminant Analysis (LDA):** Merupakan metode ekstraksi fitur *supervised* yang dirancang mengurangi dimensi sembari memaksa label komoditas agar tercerai-berai sejauh mungkin dalam area 2 dimensi (fokus melabelkan klasifikasi "benda apakah ini").
* **Principal Component Analysis (PCA):** Pemampatan metrik *unsupervised* hingga menjadi 9 Pilar Utama (PC).

### 3.2 Arsitektur Model Pengenalan Pola
* **Random Forest Regressor (Model Utama):** Algoritma *ensemble learning* ini beroperasi dengan membangun banyak pohon regresi (*decision trees*) secara paralel saat proses pelatihan. Algoritma ini dipilih karena kemampuannya yang sangat tangguh dalam melacak pola hubungan non-linear yang rumit antara fitur waktu buatan, jejak pergerakan harga historis (lag), dan variabel wilayah spasial. Prediksi akhir diperoleh melalui konsensus rata-rata dari seluruh pohon yang ada.
* **ARIMA / Autoregressive Integrated Moving Average (Model Baseline):** Algoritma peramalan statistik univariat silinder tunggal yang digunakan murni untuk membedah memori deret waktu dari harga masa lalu itu sendiri tanpa fitur tambahan luar. 

### 3.3 Parameter/Hyperparameter Model

**1. Hyperparameter Random Forest Regressor:**
* **Jumlah Pohon (`n_estimators`):** 100 (Membangun 100 pohon keputusan secara paralel untuk membangun kesimpulan akhir).
* **Seed Pengacakan (`random_state`):** 42 (Digunakan untuk mengunci pengacakan agar hasil eksperimen konsisten dan dapat direproduksi).
* **Pekerja Paralel (`n_jobs`):** -1 (Instruksi agar proses komputasi pelatihan model memaksimalkan dan menggunakan seluruh *core* CPU yang tersedia).

**2. Parameter ARIMA (Baseline):**
* **Order (p, d, q):** (1, 1, 1) *(Diasumsikan dari konvensi standar ARIMA 1,1,1 pada eksperimen pembanding)*.
  * *p (Autoregressive term)*: Menggunakan jejak waktu mundur 1 periode.
  * *d (Differencing)*: Melakukan 1 kali selisih antar waktu untuk menstabilkan tren data.
  * *q (Moving Average)*: Menggunakan batas 1 perataan kesalahan (*error window*) untuk meredam volatilitas acak.

## 4. IMPLEMENTASI DAN PENGUJIAN

### 4.1 Lingkungan Implementasi (Environment)
* **Bahasa Pemrograman:** Python.
* **Library Utama:** * **Scikit-Learn:** Digunakan untuk algoritma *Machine Learning* (`RandomForestRegressor`), reduksi dimensi (`PCA` dan `LinearDiscriminantAnalysis`), serta prapemrosesan (`StandardScaler`).
  * **Statsmodels:** Digunakan untuk memodelkan algoritma statistik deret waktu klasik (`ARIMA`).
  * **Pandas & NumPy:** Digunakan untuk manipulasi matriks, pembersihan data, dan rekayasa fitur (*feature engineering*).
  * **Matplotlib & Seaborn:** Digunakan untuk memvisualisasikan grafik EDA, *Scree Plot*, dan komparasi metrik evaluasi.
* **Perangkat Keras:** Lingkungan komputasi berbasis *Jupyter Notebook* (dapat dijalankan melalui spesifikasi laptop lokal modern atau mesin *cloud* seperti Google Colab).

### 4.2 Skenario Pengujian

Eksperimen ini dirancang ke dalam 4 skenario pemodelan utama. Untuk membongkar kecacatan metodologi evaluasi konvensional (seperti *Data Leakage* dan Bias Skala), setiap skenario model wajib melewati ujian berjenjang melalui 3 fase evaluasi berikut:

**1. Skenario Model ARIMA (Baseline)**
Sebagai model statistik klasik *univariat*, ARIMA diuji untuk melihat daya tahannya mengelola memori waktu tanpa bantuan fitur tambahan.
* **Fase 1 (Kondisi Awal):** Evaluasi awal dilakukan menggunakan sampel data rata-rata agregat nasional. Fluktuasi ekstrem tingkat provinsi sengaja diredam untuk melihat performa dasar model pada kondisi data yang sangat stabil.
* **Fase 2 (Kronologis Murni):** Dinding partisi waktu disegel. Model dilatih murni menggunakan riwayat masa lalu (2021-2024) dan diuji untuk meramal badai fluktuasi harga masa depan (2025) secara penuh lintas 38 provinsi.
* **Fase 3 (Keadilan Metrik):** Evaluasi dihitung secara terpisah dan mandiri untuk tiap kelas komoditas. Hal ini dilakukan untuk menghilangkan "Bias Skala", memastikan rentang harga komoditas elitis (seperti Daging Sapi) tidak mendistorsi ralat komoditas murah.

**2. Skenario Model Random Forest Murni**
Model ansambel ini diuji kemampuannya dalam memproses seluruh matriks fitur asli secara utuh (tanpa reduksi) untuk melacak pola non-linear.
* **Fase 1 (Skenario Data Leakage):** Data dipotong secara acak (rasio 80:20) mengabaikan urutan kalender. Fase ini sengaja memicu kebocoran data (*leakage*) dari masa depan ke masa pelatihan guna mendemonstrasikan bagaimana model bisa menghasilkan skor akurasi ilusi (nyaris 100%).
* **Fase 2 (Kronologis Murni):** Pengujian dikembalikan ke dunia nyata. Model dilarang keras melihat data 2025 saat berlatih, memaksanya murni menggunakan insting pola historis untuk menebak masa depan.
* **Fase 3 (Keadilan Metrik):** Akurasi dan ralat model (*RMSE, MAPE, $R^2$*) dihitung ulang per komoditas secara independen untuk melihat performa sejati dari algoritma hutan acak yang bebas dari polusi inflasi semu.

**3. Skenario Model PCA + Random Forest**
Skenario ini menguji dampak dari teknik reduksi dimensi *unsupervised* (PCA) terhadap memori runtut waktu sebelum dimasukkan ke model regresi.
* **Fase 1 (Skenario Data Leakage):** Sama halnya dengan RF Murni, model diuji menggunakan *split* acak yang menghasilkan performa palsu akibat bocoran jawaban dari tanggal yang berdekatan.
* **Fase 2 (Kronologis Murni):** Saat diuji meramal masa depan (2025) secara ketat, terlihat dampak negatif dari pemampatan PCA yang membuang puluhan persen varians (jejak sejarah) dari dataset, membuat model kesulitan mengenali pola.
* **Fase 3 (Keadilan Metrik):** Isolasi evaluasi per komoditas membongkar kegagalan total dari teknik pemampatan buta ini dalam meramal deret waktu harga yang berkesinambungan.

**4. Skenario Model LDA + Random Forest**
Skenario ini mengeksploitasi teknik ekstraksi dimensi *supervised* (LDA) yang dipaksa fokus pada klasifikasi label identitas komoditas.
* **Fase 1 (Skenario Data Leakage):** Pengujian acak 80:20 memberikan ilusi bahwa matriks hasil reduksi LDA masih bisa digunakan untuk menebak harga dengan baik.
* **Fase 2 (Kronologis Murni):** Ujian waktu riil membuktikan bahwa obsesi algoritma LDA untuk memisahkan nama/identitas komoditas telah membantai secara brutal informasi kontinuitas harga dan lokasi spasial.
* **Fase 3 (Keadilan Metrik):** Evaluasi spesifik per komoditas mengonfirmasi rekor metrik terburuk. Fase ini memberikan vonis akhir bahwa memaksakan algoritma pemenggal kelas (LDA) ke dalam ruang regresi fluktuasi ekonomi *time-series* adalah sebuah kecacatan metodologi yang fatal.

## 5. HASIL DAN PEMBAHASAN

### 5.1 Grafik Analitik dan Reduksi (Eksplorasi & Ekstraksi Fitur)
Evaluasi *training progress* direpresentasikan melalui analisis korelasi fitur dan perilaku algoritma reduksi dimensi sebelum regresi dilakukan:

* **Grafik Peta Panas Korelasi (Heatmap):** Menunjukkan bahwa pergerakan harga komoditas sangat terikat pada jejak historisnya. Fitur `Lag_1` (harga 1 bulan sebelumnya) mencetak skor korelasi yang sangat tinggi (0.94) terhadap harga saat ini. Ini membuktikan bahwa jejak waktu adalah roh utama dari data ini dan tidak boleh dihilangkan.
* **Scree Plot PCA:** Grafik kumulatif varians dari PCA membuktikan adanya malapetaka kehilangan informasi. Pemaksaan reduksi variabel ke dalam komponen utama (PC) membuang sebagian besar informasi historis, di mana model gagal menyerap lebih dari 40% kontinuitas data spasial dan temporal asli.
* **Dampak Reduksi LDA:** Operasi pemenggalan oleh LDA secara mutlak membantai rentang gradasi nominal harga. Algoritma ini memusatkan 100% perhitungannya hanya untuk memisahkan "identitas/kategori komoditas", sehingga merusak benang merah kronologis yang sangat dibutuhkan *Random Forest* untuk menebak tren masa depan.

### 5.2 Evaluasi Performa Model
Berdasarkan pengujian harga per komoditas per provinsi (Fase 3: Keadilan Metrik Bebas Bias Skala), di mana model diuji untuk meramal 8.711 sampel data kalender masa depan (2025) secara spesifik per komoditas, berikut adalah hasil metrik performanya:

| Model Kombinasi | MAE (Rp) | RMSE (Rp) | MAPE (%) | $R^2$ (%) |
|---|---|---|---|---|
| **S2 - Random Forest Murni** | **1,838** | **2,548** | **5.79%** | **79.28%** |
| S1 - ARIMA (Baseline) | 2,609 | 3,514 | 6.44% | 58.86% |
| S3 - PCA + Random Forest | 3,673 | 4,524 | 16.90% | 36.02% |
| S4 - LDA + Random Forest | 4,292 | 5,290 | 21.79% | 14.70% |

Perbandingan kedua model dilakukan pada Fase 3, yaitu evaluasi yang telah disesuaikan untuk menghindari data leakage dan bias skala harga, sehingga hasil yang dibandingkan benar-benar merepresentasikan kemampuan generalisasi masing-masing model pada level provinsi-komoditas yang sama.

* **Akurasi Keseluruhan (Berdasarkan $R^2$ Terbaik):** **79.28%** (Diraih oleh Random Forest Murni).
* **Analisis Kesalahan Regresi :**  
  * **Area Keberhasilan Model:** Model **Random Forest Murni** terbukti sebagai model paling andal. Tanpa menggunakan metode reduksi apapun, cabang hutannya berhasil menekan ralat (*error*) hingga persentase terendah 5.79% dengan jejak absolut kesalahan hanya Rp 2.548. Model ini sukses memetakan rumitnya pola non-linear fluktuasi tanpa terjebak *noise*. Sebagai perbandingan dasar, mesin *univariat* **ARIMA** juga tampil kokoh dan stabil ($R^2$ 58.86%) meskipun bekerja tanpa bantuan rekayasa fitur tambahan.
  * **Area Kegagalan Klasifikasi/Reduksi:** Kombinasi **LDA + RF** dan **PCA + RF** mencatatkan rekor metrik paling hancur (LDA hanya mencetak $R^2$ 14.70% dengan ralat meleset hingga 21.79%). Hal ini terjadi karena arsitektur pereduksi dimensi statis (khususnya LDA) memaksa membuang kontinuitas *time-series* demi fokus pada label identitas. Kesimpulannya, model gagal mengenali nilai harga karena dipaksa menjadi buta arah oleh reduksi dimensi.